In [10]:
import os
import pickle
import torch
import faiss
import numpy as np
from tqdm import tqdm
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.docstore.in_memory import InMemoryDocstore

In [11]:
EMBEDDING_DIM = 384  # MiniLM output dimension
N_CLUSTERS = 256  # number of IVF clusters
BATCH_SIZE = 5000
CHECKPOINT_EVERY = 10  # save every N batches

In [12]:
# load chunks
with open("../data/chunks/nepali_news_chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

print(f"Loaded {len(chunks)} chunks")

Loaded 593779 chunks


In [13]:
# load embedding model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {"device": "cuda" if torch.cuda.is_available() else "cpu"}
encode_kwargs = {"normalize_embeddings": True}

In [14]:
hf_embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 609.13it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
# embed a training sample (raw vectors)
sample_texts = [c.page_content for c in chunks[:50_000]]
# sample_vectors = np.array(hf_embeddings.embed_documents(sample_texts), dtype=np.float32)
# print(f"Training sample shape: {sample_vectors.shape}")


all_vectors = []

for i in tqdm(range(0, len(sample_texts), BATCH_SIZE), desc="Embedding training sample"):
    batch = sample_texts[i : i + BATCH_SIZE]
    batch_vectors = hf_embeddings.embed_documents(batch)
    all_vectors.extend(batch_vectors)

sample_vectors = np.array(all_vectors, dtype=np.float32)
print(f"Training sample shape: {sample_vectors.shape}")

Embedding training sample:   0%|          | 0/10 [00:00<?, ?it/s]

Embedding training sample: 100%|██████████| 10/10 [42:59<00:00, 257.92s/it]

Training sample shape: (50000, 384)


In [16]:
# build and train the IVF index
quantizer = faiss.IndexFlatL2(EMBEDDING_DIM)
index = faiss.IndexIVFFlat(quantizer, EMBEDDING_DIM, N_CLUSTERS, faiss.METRIC_L2)

In [17]:
# train the index
index.train(sample_vectors)
print(f"IVF index trained. {index.ntotal} vectors in index (training only).")

IVF index trained. 0 vectors in index (training only).


In [18]:
# wrap the trained index in a LangChain FAISS vectorstore
vectorstore = FAISS(
    embedding_function=hf_embeddings.embed_query,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)
print("Vectorstore ready with trained IVF index.")

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


Vectorstore ready with trained IVF index.


In [20]:
# create directories if they don't exist
checkpoint_dir = "../faiss_index_nepali_checkpoint"
final_index_dir = "../faiss_index_nepali"
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs(final_index_dir, exist_ok=True)

In [ ]:
# batch-insert all chunks with periodic checkpoints
num_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for i in tqdm(range(0, len(chunks), BATCH_SIZE), total=num_batches, desc="Embedding batches"):
    batch = chunks[i : i + BATCH_SIZE]
    vectorstore.add_documents(batch)

    batch_num = (i // BATCH_SIZE) + 1
    
    if batch_num % CHECKPOINT_EVERY == 0:
        vectorstore.save_local(checkpoint_dir)
        tqdm.write(f"Batch {batch_num:>4d} | chunks {i:>7d} → {i + len(batch):>7d} | Checkpoint saved | total: {index.ntotal}")
    else:
        tqdm.write(f"Batch {batch_num:>4d} | chunks {i:>7d} → {i + len(batch):>7d} | total: {index.ntotal}")

print(f"\nDone inserting. Final index size: {index.ntotal} vectors.")

# save the final index
vectorstore.save_local(final_index_dir)
print(f"Final FAISS index saved to {final_index_dir}")

Embedding batches:   0%|          | 0/119 [00:00<?, ?it/s]